In [ ]:
import pandas as pd
import os
import subprocess
import time

In [ ]:
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Calculate Ancestry MAF and Add Per Ancestry MAC from VCF

In [ ]:
Extract data for all specific ancestries (excluding OTH) in one pass
bcftools query \
  -f '%CHROM\t%POS\t%INFO/END\t%ID\t%ALT\t%INFO/AF\t%INFO/AFR_FEMALE_AC\t%INFO/AFR_MALE_AC\t%INFO/AFR_FEMALE_AN\t%INFO/AFR_MALE_AN\t%INFO/EUR_FEMALE_AC\t%INFO/EUR_MALE_AC\t%INFO/EUR_FEMALE_AN\t%INFO/EUR_MALE_AN\t%INFO/AMR_FEMALE_AC\t%INFO/AMR_MALE_AC\t%INFO/AMR_FEMALE_AN\t%INFO/AMR_MALE_AN\t%INFO/SAS_FEMALE_AC\t%INFO/SAS_MALE_AC\t%INFO/SAS_FEMALE_AN\t%INFO/SAS_MALE_AN\t%INFO/MID_FEMALE_AC\t%INFO/MID_MALE_AC\t%INFO/MID_FEMALE_AN\t%INFO/MID_MALE_AN\t%INFO/EAS_FEMALE_AC\t%INFO/EAS_MALE_AC\t%INFO/EAS_FEMALE_AN\t%INFO/EAS_MALE_AN\n' \
  AoU_srWGS_SV.v8.sites_only.vcf.gz | \
awk 'BEGIN{
    OFS="\t";
    # Header excluding OTH
    print "CHROM","POS","END","ID","ALT","GLOBAL_AF",
          "AFR_AC","AFR_AF","EUR_AC","EUR_AF","AMR_AC","AMR_AF",
          "CSA_AC","CSA_AF","MID_AC","MID_AF","EAS_AC","EAS_AF"
}
{
    # Progress Logger (prints to console, not the file)
    if (NR % 100000 == 0) { print "Processed " NR " variants..." > "/dev/stderr" }

    # Fill dots with 0 (Fields 7 through 30 for the remaining ancestries)
    for(i=7; i<=30; i++) if($i=="." || $i=="") $i=0;

    # Logic: Sum ACs, Sum ANs, Divide to get AF
    # AFR (7,8 AC | 9,10 AN)
    afr_ac=$7+$8; afr_af=(($9+$10)>0) ? afr_ac/($9+$10) : 0;
    
    # EUR (11,12 AC | 13,14 AN)
    eur_ac=$11+$12; eur_af=(($13+$14)>0) ? eur_ac/($13+$14) : 0;
    
    # AMR (15,16 AC | 17,18 AN)
    amr_ac=$15+$16; amr_af=(($17+$18)>0) ? amr_ac/($17+$18) : 0;
    
    # CSA/SAS (19,20 AC | 21,22 AN)
    csa_ac=$19+$20; csa_af=(($21+$22)>0) ? csa_ac/($21+$22) : 0;
    
    # MID (23,24 AC | 25,26 AN)
    mid_ac=$23+$24; mid_af=(($25+$26)>0) ? mid_ac/($25+$26) : 0;
    
    # EAS (27,28 AC | 29,30 AN)
    eas_ac=$27+$28; eas_af=(($29+$30)>0) ? eas_ac/($29+$30) : 0;

    # Final Print
    print $1,$2,$3,$4,$5,$6,
          afr_ac,afr_af,eur_ac,eur_af,amr_ac,amr_af,
          csa_ac,csa_af,mid_ac,mid_af,eas_ac,eas_af
}' > AoU_srWGS_SV.v8_sites_only_coordinates_Ancestry_AF_AC.tsv

echo "Master TSV (No OTH) created successfully."

# Ancestry MAC > 20 Per CNV

In [ ]:
# Define the new column mapping based on your Master TSV:
# 7:AFR_AC, 8:AFR_AF | 9:EUR_AC, 10:EUR_AF | 11:AMR_AC, 12:AMR_AF
# 13:CSA_AC, 14:CSA_AF | 15:MID_AC, 16:MID_AF | 17:EAS_AC, 18:EAS_AF

declare -A pops=( ["AFR"]=7 ["EUR"]=9 ["AMR"]=11 ["CSA"]=13 ["MID"]=15 ["EAS"]=17 )

MASTER="AoU_srWGS_SV.v8_sites_only_coordinates_Ancestry_AF_AC.tsv"

for pop in "${!pops[@]}"; do
    ac_col=${pops[$pop]}
    af_col=$((ac_col + 1))
    
    echo "Filtering $pop (AC col: $ac_col, AF col: $af_col)..."

    # Filter: Keep Header OR (AC > 21)
    # We print: CHROM, POS, END, ID, ALT, and the POP-SPECIFIC AF
    awk -v ac="$ac_col" -v af="$af_col" 'BEGIN{FS=OFS="\t"} 
    NR==1 {
        print "CHROM","POS","END","ID","ALT","AF"
    } 
    NR>1 && $ac > 21 {
        print $1, $2, $3, $4, $5, $af
    }' $MASTER > AoU_srWGS_SV.v8_sites_only_${pop}_ACgt21.tsv
done

echo "Ancestry-specific filtering complete."

###############################################################
FILTER AF <1% PER ANCESTRY, RETAIN GLOBAL AF FOR REFERENCE
###############################################################

%%bash
# Mapping: 8:AFR_AF, 10:EUR_AF, 12:AMR_AF, 14:CSA_AF, 16:MID_AF, 18:EAS_AF
# Global AF is index 6
declare -A pops=( ["AFR"]=8 ["EUR"]=10 ["AMR"]=12 ["CSA"]=14 ["MID"]=16 ["EAS"]=18 )

MASTER="AoU_srWGS_SV.v8_sites_only_coordinates_Ancestry_AF_AC.tsv"

for pop in "${!pops[@]}"; do
    af_col=${pops[$pop]}
    
    echo "Processing $pop: Comparing Global vs Ancestry-Specific AF..."

    # Filter: Keep Header OR (Ancestry AF < 0.01)
    # We print Global AF as col 6 and Ancestry AF as col 7
    awk -v af="$af_col" 'BEGIN{FS=OFS="\t"} 
    NR==1 {
        print "CHROM","POS","END","ID","ALT","GLOBAL_AF","POP_AF"
    } 
    NR>1 && $af > 0 && $af < 0.01 {
        # $6 is GLOBAL_AF from your Master TSV
        print $1, $2, $3, $4, $5, $6, $af
    }' $MASTER > AoU_srWGS_SV.v8_sites_only_${pop}_RareCNV_AncAF_GlobalAF.tsv
done

echo "Filtering complete. Global AF preserved for comparison."

# Filter AF <1% per Ancestry, Retain Global AF for Reference

In [ ]:
# Mapping: 8:AFR_AF, 10:EUR_AF, 12:AMR_AF, 14:CSA_AF, 16:MID_AF, 18:EAS_AF
# Global AF is index 6
declare -A pops=( ["AFR"]=8 ["EUR"]=10 ["AMR"]=12 ["CSA"]=14 ["MID"]=16 ["EAS"]=18 )

MASTER="./cnv_vcf_plink/AoU_srWGS_SV.v8_sites_only_coordinates_Ancestry_AF_AC.tsv"

for pop in "${!pops[@]}"; do
    af_col=${pops[$pop]}
    
    echo "Processing $pop: Comparing Global vs Ancestry-Specific AF..."

    # Filter: Keep Header OR (Ancestry AF < 0.01)
    # We print Global AF as col 6 and Ancestry AF as col 7
    awk -v af="$af_col" 'BEGIN{FS=OFS="\t"} 
    NR==1 {
        print "CHROM","POS","END","ID","ALT","GLOBAL_AF","POP_AF"
    } 
    NR>1 && $af > 0 && $af < 0.01 {
        # $6 is GLOBAL_AF from your Master TSV
        print $1, $2, $3, $4, $5, $6, $af
    }' $MASTER > ./cnv_vcf_plink/AoU_srWGS_SV.v8_sites_only_${pop}_RareCNV_AncAF_GlobalAF.tsv
done

echo "Filtering complete. Global AF preserved for comparison."

# Merge Cytoband Coordinates and Remove Duplicates

In [ ]:
# ! wget https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/cytoBand.txt.gz

# Define the ancestries you are keeping (excluding OTH if desired)
ancestries=("AFR" "EUR" "AMR" "CSA" "MID" "EAS")

for pop in "${ancestries[@]}"; do
    echo "Processing CytoBands for $pop..."

    # 1. Intersect with CytoBand
    # Input has 7 columns: CHROM, POS, END, ID, ALT, GLOBAL_AF, POP_AF
    # We skip header for bedtools (NR>1)
    awk 'BEGIN{FS=OFS="\t"} NR>1' ./cnv_vcf_plink/AoU_srWGS_SV.v8_sites_only_${pop}_RareCNV_AncAF_GlobalAF.tsv | \
    bedtools intersect -wa -wb \
        -a stdin \
        -b <(zcat ./ucsc_cytoBand/cytoBand.txt.gz) | \
    awk 'BEGIN{FS=OFS="\t"} {
        # Prints original 7 columns + the 4th column of cytoBand file (the band name)
        # In a -wb output, the b-file columns start after the 7 a-file columns.
        # b-column 4 is therefore index 7 + 4 = 11.
        print $1, $2, $3, $4, $5, $6, $7, $11
    }' > ./cnv_vcf_plink/${pop}_temp_with_dups.bed

    # 2. Collapse multiple cytobands (Deduplication)
    # Key = Chr(1), Start(2), End(3), ID(4), Alt(5), GlobalAF(6), PopAF(7)
    awk -F'\t' 'BEGIN{OFS="\t"} {
        key = $1 FS $2 FS $3 FS $4 FS $5 FS $6 FS $7; 
        if (key in a) { 
            # Append band ($8) if it is not already there
            if (index(a[key], $8) == 0) {
                a[key] = a[key] ";" $8; 
            }
        } else { 
            a[key] = $0; 
        } 
    } END { 
        for (i in a) print a[i]; 
    }' ./cnv_vcf_plink/${pop}_temp_with_dups.bed > ./cnv_vcf_plink/AoU_srWGS_SV.v8_sites_only_${pop}_RareCNV_cytoband_unique.bed

    # Clean up
    rm ./cnv_vcf_plink/${pop}_temp_with_dups.bed
done

echo "CytoBand merging and removing duplicates complete."

In [ ]:
# Remove duplications
#v8
!awk -F'\t' \
    '{ key = $1 FS $2 FS $3 FS $4 FS $5 FS $6; if (key in a) { a[key] = a[key] ";" $10; } else { a[key] = $0; } } END { for (i in a) print a[i]; }' cnv_vcf_plink/AoU_srWGS_SV.v8_sites_only_coordinates_AF_cytoband.bed \
    > cnv_vcf_plink/AoU_srWGS_SV.v8_sites_only_coordinates_AF_cytoband_unique.bed

# Create GENECODE Map to Genes

In [ ]:
zcat genecode/gencode.v46.annotation.gff3.gz | awk 'BEGIN { OFS="\t" } $3 == "gene" {
    # Split the attributes column ($9) by semicolon
    split($9, a, ";"); 
    
    gene_name = ""; 
    gene_id = "";
    
    # Loop through the split attributes
    for (i in a) { 
        # Extract Gene Symbol (strips "gene_name=" which is 10 characters long)
        if (a[i] ~ /gene_name=/) { 
            gene_name = substr(a[i], 11); 
        } 
        # Extract Ensembl Gene ID (strips "gene_id=" which is 8 characters long)
        if (a[i] ~ /gene_id=/) { 
            gene_id = substr(a[i], 9); 
        }
    } 
    
    # Print the desired columns only if both ID and Symbol are found
    if (gene_name != "" && gene_id != "") { 
        # $1 (Chr), $4-1 (Start 0-based), $5 (End 1-based), Gene Symbol, Gene ID
        print $1, $4-1, $5, gene_name, gene_id; 
    } 
}' > genecode/gencode_gene_map.tsv

# Intersect GENECODE Map to Ancestry Specific Annotation Tables

In [ ]:
# Define ancestries (adjust list as needed)
ancestries=("AFR" "EUR" "AMR" "CSA" "MID" "EAS")

for pop in "${ancestries[@]}"; do
    echo "Processing $pop with Gencode..."
    
    # 1. Sort the input file (8 columns: chr, pos, end, id, alt, global_af, pop_af, cytoband)
    sort -k1,1 -k2,2n ./cnv_vcf_plink/AoU_srWGS_SV.v8_sites_only_${pop}_RareCNV_cytoband_unique.bed \
        > ./cnv_vcf_plink/${pop}_temp_sorted.bed

    # 2. Run intersection
    # -a has 8 columns
    # -b (gencode_genes_SORTED.bed) has 4 columns: chr, start, end, gene_name
    bedtools intersect -wa -wb -loj \
        -a ./cnv_vcf_plink/${pop}_temp_sorted.bed \
        -b ./genecode/gencode_genes_SORTED.bed \
        > ./cnv_vcf_plink/${pop}_temp_genes_dups.bed

    # 3. Collapse multiple genes
    # The gene name is the 4th column of file B. 
    # Since file A has 8 columns, the gene name is at index 8 + 4 = 12.
    awk -F'\t' 'BEGIN{OFS="\t"} {
        # Key now covers all 8 columns from the cytoband file
        key = $1 FS $2 FS $3 FS $4 FS $5 FS $6 FS $7 FS $8;
        gene = $12; 
        
        if (key in a) {
            # If gene is not a dot and not already in our list for this CNV, append it
            if (gene != "." && index(a[key], gene) == 0) {
                a[key] = a[key] ";" gene;
            }
        } else {
            a[key] = $0;
        }
    } END { for (i in a) print a[i]; }' \
    ./cnv_vcf_plink/${pop}_temp_genes_dups.bed | \
    sort -k1,1 -k2,2n > ./cnv_vcf_plink/AoU_srWGS_SV.v8_sites_only_${pop}_RareCNV_final.bed
    
    # Clean up intermediate files
    rm ./cnv_vcf_plink/${pop}_temp_genes_dups.bed
    rm ./cnv_vcf_plink/${pop}_temp_sorted.bed
done

echo "Final annotation and sorting complete for all ancestries."

In [ ]:
# INTERGENIC VS GENIC rCNV COUNTS PER ANCESTRY

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Define the ancestries you want to visualize
ancestries = ["EUR", "AFR", "AMR"]
output_dir = "./plots_and_tables/"
os.makedirs(output_dir, exist_ok=True)

for pop in ancestries:
    file_path = f"./cnv_vcf_plink/AoU_srWGS_SV.v8_CNVs_{pop}_final_master_annotation.tsv"
    
    if os.path.exists(file_path):
        # 1. Load Data
        df_pop = pd.read_csv(file_path, sep="\t")
        print(f"\n--- Processing {pop} ({len(df_pop)} variants) ---")
        
        # 2. Faster Categorization (Vectorized)
        df_pop['Region_Type'] = np.where(df_pop['collapsed_genes'] == ".", 'Intergenic', 'Genic')

        # 3. Stats for labels
        order = ['Intergenic', 'Genic']
        counts = df_pop['Region_Type'].value_counts().reindex(order).fillna(0)
        total = len(df_pop)
        percentages = (counts / total) * 100

        # 4. Plotting
        plt.figure(figsize=(7, 5))
        ax = sns.countplot(data=df_pop, x='Region_Type', order=order, 
                           palette={'Genic': 'salmon', 'Intergenic': 'lightgray'})

        # Add percentage labels
        for i, p in enumerate(ax.patches):
            height = p.get_height()
            ax.text(p.get_x() + p.get_width()/2., height + (total * 0.01),
                    f'{percentages[i]:.1f}%', ha="center", fontweight='bold')

        plt.title(f'{pop} rCNVs: Proportion of Genic vs. Intergenic', fontsize=13)
        plt.ylim(0, max(counts) * 1.15)
        
        # --- SAVE THE PLOT ---
        plot_name = f"{output_dir}{pop}_genic_distribution.png"
        plt.savefig(plot_name, dpi=300, bbox_inches='tight')
        plt.show() # Show it in the notebook
        plt.close() # Close to save memory
        
        # --- SAVE THE UPDATED TABLE (Optional) ---
        # Only do this if you want the 'Region_Type' column saved permanently
        # df_pop.to_csv(f"./cnv_vcf_plink/{pop}_with_region_type.tsv", sep="\t", index=False)
        
        print(f"Saved plot to {plot_name}")
    else:
        print(f"File for {pop} not found. Skipping...")

In [ ]:
# END